# v0: within-channel packaging ranker (frozen embeddings + LightGBM)

**Question:** given a channel's uploads, how well do title and thumbnail alone predict which videos beat that channel's own norm?

- **Target:** `y`, the robust z-score (median, 1.4826 x MAD) of age-adjusted log views *within each channel*. The age adjustment is a pooled within-channel slope of log views on log age, fit on training channels only. Channel size cancels out, so the model can only learn packaging and format.
- **Features:** title and format metadata, frozen title embeddings (`BAAI/bge-base-en-v1.5`), frozen SigLIP 2 NaFlex thumbnail embeddings at full size and at 168x94 (the phone-legibility test), and simple image statistics.
- **Model:** LightGBM regression, early-stopped on validation channels. There is also a `lambdarank` variant.
- **Metric:** mean per-channel Spearman correlation on (a) **held-out channels** and (b) the **newest 120 days** of the remaining channels, with bootstrap 95% CIs over channels.

**Data:** the latest full collector run under `$YT_DATA/raw` (mounted read-only at `/data/yt`). Uploads after 2026-08-24 are excluded, because YouTube changed how it counts views on that date.

**Policy:** this dataset comes from YouTube Data API v3 snapshots held privately. See the collector README's policy section before sharing any data or derived metrics.

**Knobs (env vars):** `LAB_SAMPLE_CHANNELS` (subsample for quick runs), `LAB_SKIP_IMAGES=1` (titles only), `LAB_TEXT_MODEL`, `LAB_IMAGE_MODEL`.

## 0. Setup

This cell loads the helper module `lab/ytlab.py` (the notebook's logic lives there so it can be unit-tested), resolves paths, and reads the environment knobs.

- `DATA` is the collector output, mounted read-only into the container at `/data/yt`.
- `CACHE` holds downloaded thumbnails and computed embeddings, so later runs skip that work. `ART` holds result files. Git ignores both folders.
- `MIN_VIDEOS = 20`: a channel needs at least 20 qualifying uploads before its median and spread mean anything. Channels with fewer are dropped.
- `MIN_AGE_DAYS = 30`: videos younger than a month are still gaining views quickly, so they are left out of the targets.
- `PCA_DIM = 64`: embeddings are compressed to 64 numbers each before LightGBM sees them (step 3 explains why).

**Check:** the printed device should be `cuda` with the GB10. If it says `cpu`, the container can't see the GPU; run `python lab/smoke.py` to diagnose.

In [ ]:
import os, sys, json, time
from pathlib import Path
import numpy as np, pandas as pd

REPO = Path(os.environ.get("LAB_REPO", "/workspace"))
sys.path.insert(0, str(REPO / "lab"))
import ytlab

DATA = Path(os.environ.get("YT_DATA", "/data/yt"))
CACHE = Path(os.environ.get("LAB_CACHE", REPO / "lab" / ".cache")); CACHE.mkdir(parents=True, exist_ok=True)
ART = Path(os.environ.get("LAB_ARTIFACTS", REPO / "lab" / "artifacts")); ART.mkdir(parents=True, exist_ok=True)
SAMPLE_CHANNELS = int(os.environ.get("LAB_SAMPLE_CHANNELS", "0")) or None
SKIP_IMAGES = os.environ.get("LAB_SKIP_IMAGES", "0") == "1"
TEXT_MODEL = os.environ.get("LAB_TEXT_MODEL", "BAAI/bge-base-en-v1.5")
IMAGE_MODEL = os.environ.get("LAB_IMAGE_MODEL", "google/siglip2-base-patch16-naflex")
SEED, MIN_VIDEOS, MIN_AGE_DAYS, PCA_DIM = 0, 20, 30, 64

import warnings
warnings.filterwarnings("ignore", message=".*eval_set.*deprecated.*")
warnings.filterwarnings("ignore", message=".*eval_group.*deprecated.*")

import torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("torch", torch.__version__, "| device", DEVICE, torch.cuda.get_device_name(0) if DEVICE == "cuda" else "")

## 1. Load the latest full run

The collector writes one folder per run. This picks the latest **complete full** run, which has every upload in the 24-month window with titles and thumbnails. The 3-channel smoke test is skipped automatically. If no full run has finished yet, the fullest paused run is used and the printout says so.

`videos` has one row per upload; `channels` carries each channel's bucket (niche, adjacent, far) from the tracker.

**Check:** the channel count should be close to the number in `channels.csv` (967 on the first pass), with about 200k video rows.

In [ ]:
runs = ytlab.list_runs(DATA)
run = ytlab.pick_full_run(runs)
print(f"using run {run['run_id']} status={run.get('status')} channels={len(run.get('completed', []))}/{run.get('n_channels')}")
videos, channels = ytlab.load_run(run)
print(f"{len(videos):,} video rows, {len(channels):,} channel rows")
bucket = channels.set_index("channel_id")["bucket"] if "bucket" in channels else pd.Series(dtype=str)

## 2. Filter, split, target

Three things happen here.

**1. Filtering** (`ytlab.prepare`) keeps only videos the target is meaningful for:
- long-form (over 180 s, the project's Shorts cutoff), not live streams or premieres, and public;
- at least 30 days old;
- uploaded before 2026-08-24, when YouTube started counting views from the first frame. Mixing videos from before and after that change would compare different units;
- from channels with at least 20 such videos.

**2. Splits** (`ytlab.assign_splits`), decided by a stable hash of the channel ID so they are the same on every run:
- `test_channel`: 20% of channels, held out entirely. Answers "does this work on a channel the model has never seen?"
- `val`: 10% of channels, used only to decide when training stops.
- `test_time`: the newest 120 days of uploads from the remaining channels. Answers "does what worked before still work now?"
- `train`: everything else.

**3. The target `y`** (`ytlab.add_target`):
- Older videos have more views just from being up longer. The age slope is fit within channels (each channel compared with its own average), on training channels only. Each video's log views then have `slope x log(age)` subtracted.
- The adjusted value is turned into a robust z-score within its own channel: (value minus the channel's median) / (1.4826 x the channel's median absolute deviation). `y = 0` is a typical video for that channel; `y = 2` is a strong outlier for that channel. That holds whether the channel has 5k or 5M subscribers, so channel size can't leak into the model.
- `multiple` is the same quantity on the familiar scale: age-adjusted views divided by the channel median.

**Check:** the slope should be positive and below 1. `y` should be centered at 0 with spread near 1, and each split should have a reasonable number of channels.

In [ ]:
df = ytlab.prepare(videos, min_age_days=MIN_AGE_DAYS, min_videos=MIN_VIDEOS)
if SAMPLE_CHANNELS:
    keep = pd.Series(df["channel_id"].unique()).sample(min(SAMPLE_CHANNELS, df["channel_id"].nunique()), random_state=SEED)
    df = df[df["channel_id"].isin(set(keep))].reset_index(drop=True)
df["bucket"] = df["channel_id"].map(bucket).fillna("unknown")
df = ytlab.assign_splits(df, seed=SEED)
slope = ytlab.fit_age_slope(df, df["split"] == "train")
df = ytlab.add_target(df, slope)
print(f"age slope (log views per log day, within channel): {slope:.3f}")
print(df.groupby("split").agg(videos=("video_id", "size"), channels=("channel_id", "nunique")))
print(df.groupby("bucket")["channel_id"].nunique().rename("channels"))
df["y"].describe()

Two sanity plots. **Left:** the distribution of `y`. It should be roughly bell-shaped around 0, with a longer right tail, because outliers are mostly upside. **Right:** channel-demeaned log views against log age, with the fitted slope. If the line clearly misses the cloud, the age adjustment is wrong and every later number is suspect.

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
ax[0].hist(df["y"], bins=80); ax[0].set_title("target y (within-channel robust z)"); ax[0].set_xlabel("y")
s = df.sample(min(len(df), 20000), random_state=SEED)
ax[1].scatter(s["log_age"], s["log_views"] - s.groupby("channel_id")["log_views"].transform("mean"), s=2, alpha=.2)
xs = np.linspace(s["log_age"].min(), s["log_age"].max(), 50)
ax[1].plot(xs, slope * (xs - s["log_age"].mean()), lw=2)
ax[1].set_xlabel("log age (days)"); ax[1].set_ylabel("log views, channel-demeaned"); ax[1].set_title("age adjustment")
plt.tight_layout(); plt.show()

## 3. Features

**Metadata features** are cheap and readable: title length in characters and words; whether the title has a digit, question mark, exclamation, dollar sign, "vs" or brackets; capital-letter ratio and emoji count; log duration; publish hour and weekday (UTC). These are the features you could reason about by hand. Every richer feature set has to beat them.

**Title embeddings:** each title is turned into a 768-number vector by `BAAI/bge-base-en-v1.5`, a frozen sentence encoder, so titles that mean similar things get similar vectors. "Frozen" means the encoder is used as-is, not trained here. Vectors are cached per video ID and title, so a retitled video (from an A/B test) gets a fresh embedding.

**Check:** the shape should be (number of videos, 768). The first run downloads the model (about 400 MB) to the Hugging Face cache.

In [ ]:
meta = ytlab.title_features(df)
t0 = time.time()
title_emb = ytlab.embed_titles(df, TEXT_MODEL, DEVICE, CACHE)
print("title embeddings", title_emb.shape, f"{time.time()-t0:.0f}s")

**Thumbnails.** Each video's highest-resolution thumbnail is downloaded once to `lab/.cache/thumbs/`, 16 at a time, and reused on later runs. Then:

- **Full-size embedding:** SigLIP 2 NaFlex (`google/siglip2-base-patch16-naflex`) turns each thumbnail into a 768-number vector. NaFlex keeps the 16:9 shape (padding instead of cropping), so text at the edges of a thumbnail isn't cut off. SigLIP 2 was trained to read text in images, which matters because thumbnail text is a big part of packaging.
- **168x94 embedding:** the same thumbnail is first shrunk to roughly phone-feed size, then embedded. This tests the project rule that a thumbnail must read at 120px: if a thumbnail's meaning survives shrinking, the two embeddings stay similar.
- **Image statistics:** brightness, contrast, saturation and colorfulness on a tiny render. Crude, but readable.

This is the slow cell on the first run, about 200k thumbnails embedded twice. On the GB10 expect minutes to tens of minutes, plus download time. For a quick pass, set `LAB_SAMPLE_CHANNELS=100`.

**Check:** most thumbnails should download. A few failures (deleted or private videos) are fine; those rows get zero vectors and `has_thumb = 0`.

In [ ]:
if SKIP_IMAGES:
    img_full = img_small = None; img_mask = np.zeros(len(df), bool); stats = pd.DataFrame(index=df.index)
else:
    t0 = time.time()
    paths = ytlab.download_thumbnails(df, CACHE / "thumbs")
    print(f"thumbnails: {paths.notna().sum():,}/{len(df):,} in {time.time()-t0:.0f}s")
    t0 = time.time()
    img_full, img_mask = ytlab.embed_images(paths.tolist(), IMAGE_MODEL, DEVICE, CACHE, tag="full")
    img_small, _ = ytlab.embed_images(paths.tolist(), IMAGE_MODEL, DEVICE, CACHE, tag="168x94", small=(168, 94))
    stats = ytlab.image_stats(paths.tolist()); stats.index = df.index
    print("image embeddings", img_full.shape, f"{time.time()-t0:.0f}s")

**Compress the embeddings, then assemble the feature sets.**

LightGBM handles hundreds of raw embedding dimensions poorly at this data size; it overfits to noise in individual dimensions. PCA keeps the 64 directions that vary most. It is fit on training rows only, so the test sets don't shape the features. The printout shows how much of each embedding's variance survives.

The five feature sets are cumulative comparisons:
1. **naive:** title length + duration. The floor to beat.
2. **meta:** all the hand-built title and format features.
3. **meta + title:** adds title meaning.
4. **meta + thumbnail:** adds the thumbnail embeddings (full and small) and image statistics.
5. **all:** everything.

Comparing 2 with 3 tells you whether title wording adds signal beyond surface features. Comparing 2 with 4 tells you the same for the thumbnail.

In [ ]:
from sklearn.decomposition import PCA
tr = (df["split"] == "train").values

def reduce(name, X):
    if X is None:
        return pd.DataFrame(index=df.index)
    p = PCA(n_components=min(PCA_DIM, X.shape[1]), random_state=SEED).fit(X[tr])
    print(f"{name}: PCA {PCA_DIM} keeps {p.explained_variance_ratio_.sum():.0%} of variance")
    return pd.DataFrame(p.transform(X), index=df.index, columns=[f"{name}_{i}" for i in range(p.n_components_)])

T = reduce("title", title_emb)
I = reduce("thumb", img_full)
S = reduce("thumb168", img_small)
I["has_thumb"] = img_mask.astype(int) if not SKIP_IMAGES else 0

FEATURE_SETS = {
    "naive (title length + duration)": meta[["t_chars", "log_duration"]],
    "meta": meta,
    "meta + title": pd.concat([meta, T], axis=1),
}
if not SKIP_IMAGES:
    FEATURE_SETS["meta + thumbnail"] = pd.concat([meta, stats, I, S], axis=1)
    FEATURE_SETS["all"] = pd.concat([meta, stats, T, I, S], axis=1)
{k: v.shape[1] for k, v in FEATURE_SETS.items()}

## 4. Train and evaluate

**Train one LightGBM regressor per feature set** to predict `y`. Settings are conservative: small learning rate, half the columns sampled per tree, at least 40 videos per leaf. Training stops when the validation channels stop improving (up to 4,000 trees, patience 200).

**Evaluation:** for each held-out channel, rank its videos by the model's prediction and by the true `y`, then take the Spearman correlation between the two rankings. Average over channels. Channels with fewer than 10 test videos are skipped. This matches how you'd use the model: ranking your own options against each other, not predicting raw views.

How to read the correlation `rho`: 0 is no better than chance, 0.1 to 0.2 is weak but real signal, 0.3 or more is strong for this kind of problem.

In [ ]:
import lightgbm as lgb

def fit_regressor(X):
    m = lgb.LGBMRegressor(n_estimators=4000, learning_rate=0.03, num_leaves=31, min_child_samples=40,
                          subsample=0.8, subsample_freq=1, colsample_bytree=0.5, reg_lambda=1.0,
                          random_state=SEED, verbose=-1)
    va = (df["split"] == "val").values
    m.fit(X[tr], df.loc[tr, "y"], eval_set=[(X[va], df.loc[va, "y"])],
          callbacks=[lgb.early_stopping(200, verbose=False)])
    return m

results, preds, models = [], {}, {}
for name, X in FEATURE_SETS.items():
    t0 = time.time()
    m = fit_regressor(X)
    p = m.predict(X)
    preds[name], models[name] = p, m
    d = df.assign(pred=p)
    for split in ["test_channel", "test_time"]:
        r = ytlab.within_channel_spearman(d[d["split"] == split], "pred")
        results.append({"features": name, "split": split, **r, "trees": m.best_iteration_, "secs": round(time.time() - t0)})
res = pd.DataFrame(results)
res.pivot(index="features", columns="split", values="rho").round(3)

**The results table with uncertainty.** Each cell reads `rho [low, high] n=channels`. The bracket is a 95% bootstrap confidence interval: channels are resampled with replacement 1,000 times. If two feature sets' intervals overlap heavily, treat them as tied. Results are also saved to `lab/artifacts/results_v0.json` so later runs can be compared.

In [ ]:
fmt = res.assign(ci=lambda r: r.apply(lambda x: f"{x.rho:.3f} [{x.lo:.3f}, {x.hi:.3f}] n={x.channels}", axis=1))
display(fmt.pivot(index="features", columns="split", values="ci"))
(ART / "results_v0.json").write_text(json.dumps({
    "run_id": run["run_id"], "slope": slope, "n_videos": int(len(df)), "n_channels": int(df["channel_id"].nunique()),
    "text_model": TEXT_MODEL, "image_model": None if SKIP_IMAGES else IMAGE_MODEL,
    "sample_channels": SAMPLE_CHANNELS, "results": results}, indent=2, default=float))
print("saved", ART / "results_v0.json")

### By bucket (held-out channels, best feature set)

Does packaging signal differ between your niche, adjacent tech and far-out channels? If niche is much weaker than far, the model is mostly learning general YouTube packaging rather than what works in your niche. That's useful, but it's a different tool. Small buckets have wide intervals; read them with that in mind.

In [ ]:
best = res[res.split == "test_channel"].sort_values("rho", ascending=False).iloc[0]["features"]
d = df.assign(pred=preds[best])
pd.DataFrame({b: ytlab.within_channel_spearman(g, "pred") for b, g in d[d.split == "test_channel"].groupby("bucket")}).T.round(3)

### Which feature groups the best model uses

Total split gain per feature group, as a share of the whole. `title` and `thumb`/`thumb168` are the embedding components, `img` the image statistics, `t` the hand-built title features, `log`/`pub` duration and publish time. This shows where the model found signal. It is not causal: a feature can carry gain because it correlates with something else.

In [ ]:
m = models[best]
imp = pd.Series(m.booster_.feature_importance("gain"), index=m.booster_.feature_name())
group = imp.index.str.extract(r"^(title|thumb168|thumb|img|t|log|pub|has)", expand=False).fillna("other")
imp.groupby(group.values).sum().sort_values(ascending=False).div(imp.sum()).round(3)

## 5. Ranking objective (lambdarank) on the best feature set

The regressor tries to predict `y` exactly. A ranker only cares about order within each channel, which is closer to the real use (picking between title or thumbnail options). Here each channel's videos are graded 0 to 4 by quintile of `y`, and LightGBM's `lambdarank` learns to put higher grades first, with each channel as its own group.

If the ranker beats the regressor clearly, use ranking objectives from here on. If they tie, the regressor is simpler to reason about.

In [ ]:
X = FEATURE_SETS[best]
lab = df.groupby("channel_id")["y"].transform(lambda s: pd.qcut(s.rank(method="first"), 5, labels=False)).astype(int)
order = np.argsort(df["channel_id"].values, kind="stable")
def groups(mask):
    idx = order[mask[order]]
    return idx, df["channel_id"].values[idx]
tri, trg = groups(tr); vai, vag = groups((df["split"] == "val").values)
sizes = lambda g: pd.Series(g).groupby(g, sort=False).size().values
rk = lgb.LGBMRanker(objective="lambdarank", n_estimators=3000, learning_rate=0.03, num_leaves=31,
                    min_child_samples=40, subsample=0.8, subsample_freq=1, colsample_bytree=0.5,
                    random_state=SEED, verbose=-1)
rk.fit(X.values[tri], lab.values[tri], group=sizes(trg),
       eval_set=[(X.values[vai], lab.values[vai])], eval_group=[sizes(vag)], eval_at=[5],
       callbacks=[lgb.early_stopping(200, verbose=False)])
d = df.assign(pred=rk.predict(X.values))
pd.DataFrame({s: ytlab.within_channel_spearman(d[d.split == s], "pred") for s in ["test_channel", "test_time"]}).T.round(3)

## 6. Sanity check: best and worst predicted videos on held-out channels

Numbers can hide nonsense. For the three largest held-out channels, this shows the 3 videos the model ranks highest and the 3 it ranks lowest, with true views, age, `multiple` and `y`. Read the titles. Do the top picks look like better packaging, or did the model latch onto something incidental, like a series name or upload era?

In [ ]:
d = df.assign(pred=preds[best])
test = d[d.split == "test_channel"]
pick = test.groupby("channel_id").size().sort_values(ascending=False).index[:3]
for c in pick:
    g = test[test.channel_id == c].sort_values("pred", ascending=False)
    print(f"\n== {c} ({g.bucket.iloc[0]}), {len(g)} videos")
    show = pd.concat([g.head(3), g.tail(3)])[["title", "view_count", "age_days", "multiple", "y", "pred"]]
    display(show.round(2))

## Reading the results

- **Baseline to beat:** `naive`. If `meta + title` or `all` doesn't clear it with non-overlapping CIs on held-out channels, packaging signal in frozen embeddings is weak at this data size. Collect more data before fine-tuning anything.
- **`test_time` vs `test_channel`:** a model that does well on held-out channels but poorly forward in time has learned what worked historically rather than what works now. That matters for choosing titles today.
- **Leakage note:** the per-channel median and MAD use all of a channel's qualifying videos, including its `test_time` uploads. The age slope is fit on training channels only.
- **Next:** daily refresh snapshots allow fitting the target at a fixed age (views at day 30) instead of a single cross-section. After that, try fine-tuning the text or vision encoder with a pairwise loss if v0 shows signal.